In [ ]:
import sys
sys.path.append('..')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from argparse import Namespace

from scene import Scene
from scene.gaussian_model import GaussianModel
from gaussian_renderer import render, render_env_map
from utils.loss_utils import ssim
from utils.image_utils import psnr
from lpipsPyTorch import LPIPS

torch.set_grad_enabled(False)

lpips_fn = LPIPS(net_type='vgg').cuda()

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
MODEL_PATH  = '../output/refnerf/toaster'
SOURCE_PATH = '../data/refnerf/toaster'
N_VIEWS     = 6    # test views to display per iteration (all are used for metrics)
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
def get_saved_iterations(model_path):
    pc_dir = os.path.join(model_path, 'point_cloud')
    iters = []
    if os.path.exists(pc_dir):
        for d in os.listdir(pc_dir):
            if d.startswith('iteration_'):
                iters.append(int(d.split('_')[1]))
    return sorted(iters)

iterations = get_saved_iterations(MODEL_PATH)
print('Saved iterations:', iterations)

In [ ]:
def load_model(model_path, source_path, iteration):
    model_args = Namespace(
        sh_degree=3,
        source_path=os.path.abspath(source_path),
        model_path=model_path,
        images='images',
        resolution=1,
        white_background=False,
        data_device='cuda',
        eval=True,
        albedo_bias=0,
        cubemap_resolution=512,
    )
    pipe_args = Namespace(depth_ratio=0.0, debug=False,
                          convert_SHs_python=False, compute_cov3D_python=False)
    gaussians = GaussianModel(model_args.sh_degree, model_args)
    scene = Scene(model_args, gaussians, load_iteration=iteration, shuffle=False)
    return gaussians, scene, pipe_args

In [ ]:
bg = torch.zeros(3, device='cuda')

for iteration in iterations:
    print(f'\n=== Iteration {iteration} ===')
    gaussians, scene, pipe = load_model(MODEL_PATH, SOURCE_PATH, iteration)
    test_cams = scene.getTestCameras()

    # ── Render all test views ────────────────────────────────────────────────
    all_renders, all_normals, all_gts = [], [], []
    psnr_sum = ssim_sum = lpips_sum = 0.0

    for cam in test_cams:
        pkg = render(cam, gaussians, pipe, bg, env_start_iter=None, iteration=iteration)
        alpha = pkg['rend_alpha']

        pred  = (pkg['pbr_rgb'] * alpha + (1 - alpha)).clamp(0, 1)          # [3,H,W]
        norm  = ((pkg['rend_normal'] + 1) / 2 * alpha + (1 - alpha)).clamp(0, 1)

        gt_rgba = cam.original_image.cuda()
        gt = (gt_rgba[:3] * gt_rgba[3:] + (1 - gt_rgba[3:])).clamp(0, 1)   # [3,H,W]

        psnr_sum  += psnr(pred[None], gt[None]).mean().item()
        ssim_sum  += ssim(pred[None], gt[None]).mean().item()
        lpips_sum += lpips_fn(pred[None], gt[None]).mean().item()

        all_renders.append(pred.permute(1, 2, 0).cpu().numpy())
        all_normals.append(norm.permute(1, 2, 0).cpu().numpy())
        all_gts.append(gt.permute(1, 2, 0).cpu().numpy())

    n = len(test_cams)
    print(f'  PSNR  {psnr_sum/n:.2f}  |  SSIM  {ssim_sum/n:.4f}  |  LPIPS  {lpips_sum/n:.4f}')

    # ── Display N_VIEWS evenly-spaced views ──────────────────────────────────
    idxs = np.linspace(0, n - 1, N_VIEWS, dtype=int)
    fig, axes = plt.subplots(3, N_VIEWS, figsize=(3 * N_VIEWS, 9))
    fig.suptitle(f'Iteration {iteration}  |  PSNR {psnr_sum/n:.2f}  SSIM {ssim_sum/n:.4f}  LPIPS {lpips_sum/n:.4f}', fontsize=12)
    for row, (imgs, label) in enumerate([(all_gts, 'GT'), (all_renders, 'Render'), (all_normals, 'Normal')]):
        for col, i in enumerate(idxs):
            ax = axes[row, col]
            ax.imshow(np.clip(imgs[i], 0, 1))
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(label, fontsize=11)
    plt.tight_layout()
    plt.show()

    # ── Environment map ──────────────────────────────────────────────────────
    if gaussians.envmap is not None:
        env_res = render_env_map(gaussians)
        env_img = env_res['env_cood1'].permute(1, 2, 0).cpu().numpy()
        fig2, ax2 = plt.subplots(figsize=(12, 4))
        ax2.imshow(np.clip(env_img, 0, 1))
        ax2.set_title(f'Environment Map — Iteration {iteration}')
        ax2.axis('off')
        plt.tight_layout()
        plt.show()

    del gaussians, scene
    torch.cuda.empty_cache()